# 01: Data audit

Working notes for an evidence-based look at the Pump It Up data before cleaning, feature engineering or modelling.

I need to write the code, inspect the results and make the calls. These prompts are here to stop the audit becoming a tour of whichever columns happen to look interesting.

## Ground rule

Do not touch the source frames here. Count a problem before trying to fix it, record the evidence, then test the treatment in the baseline workflow.

Keep three statements separate for each feature:

- **Observation:** a fact supported by a count, summary or plot.
- **Interpretation:** my explanation of that fact.
- **Decision to test:** the treatment I plan to evaluate in the baseline.

Otherwise a hunch will become a cleaning rule, and I will forget why.

## Things I need to establish

1. Do the files have the expected structure and matching identifiers?
2. Which values represent genuine measurements, explicit nulls or hidden missing-data markers?
3. Does the target distribution create an imbalance problem?
4. Which features need transformation, grouping or exclusion?
5. Do related features describe the same underlying concept?
6. Does the test set differ from the training set in ways the preprocessing must handle?

## 1. Set up the notebook

Import the libraries needed for tabular analysis and plotting. Keep paths relative to the project; nobody else has my directory layout.

Expected local files:

- `TrainingSetValues.csv`
- `TrainingSetLabels.csv`
- `TestSetValues.csv`
- `SubmissionFormat.csv`

In [55]:
# Imports and project-relative data paths go here.
# Check that all four files exist before carrying on.

# Imports
print("Importing libraries...")
import pandas as pd
from pathlib import Path
print("Imports completed")
print("Pandas Version: ", pd.__version__)

# Data file presence checks
print("Confirming presence of data files...")
training_set_values_path = Path("../data/TrainingSetValues.csv")
training_set_labels_path = Path("../data/TrainingSetLabels.csv")
test_set_values_path = Path("../data/TestSetValues.csv")
submission_format_path = Path("../data/SubmissionFormat.csv")

loadPaths = [
    training_set_values_path,
    training_set_labels_path,
    test_set_values_path,
    submission_format_path
]

for path in loadPaths:
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

    print(f"{path} - file found OK")

print("Data file checks complete")



Importing libraries...
Imports completed
Pandas Version:  3.0.2
Confirming presence of data files...
..\data\TrainingSetValues.csv - file found OK
..\data\TrainingSetLabels.csv - file found OK
..\data\TestSetValues.csv - file found OK
..\data\SubmissionFormat.csv - file found OK
Data file checks complete


## 2. Load the four source files

One DataFrame per file. Keep the training values and labels separate until the identifier check passes.

Pick dull, consistent variable names. Future me will cope.

In [56]:
# Load the training values, training labels, test values and submission format.
training_set_values = pd.read_csv(training_set_values_path)
training_set_labels = pd.read_csv(training_set_labels_path)
test_set_values = pd.read_csv(test_set_values_path)
submission_format = pd.read_csv(submission_format_path)

print("Data loaded ok")
print("==============")
print("training_set_values:", training_set_values.shape[0])
print("training_set_labels:", training_set_labels.shape[0])
print("test_set_values:", test_set_values.shape[0])
print("submission_format:", submission_format.shape[0])

Data loaded ok
training_set_values: 59400
training_set_labels: 59400
test_set_values: 14850
submission_format: 14850


## 3. Check file structure and alignment

Checks to write for each DataFrame:

- row and column counts;
- column names and data types;
- duplicate identifiers;
- whether training IDs match label IDs;
- whether test IDs match the submission template;
- whether train and test contain the same predictor columns.

A failed alignment check stops the audit. There is little value analysing rows that may not belong together.

In [73]:
# File-structure and identifier check helpers.

# Sanity check whether two input DFs pair as a valid 'predictors' / 'labels' pair
# Kept it generic, rather than implementation-dependent. 'ID' column is required in both DFs
def compare_values_and_labels(values_df, labels_df, allow_unordered: bool = False):
    """Check that predictor and label frames contain compatible IDs.

    Args:
        values_df: DataFrame containing predictor rows and an ``id`` column.
        labels_df: DataFrame containing label rows and an ``id`` column.
        allow_unordered: Accept matching ID sets in different row orders. The
            default requires IDs to match in the same order.
    """
    if not isinstance(allow_unordered, bool):
        raise TypeError("allow_unordered must be a bool")

    # Check that both arguments are DataFrames.
    for name, df in {
        "values_df": values_df,
        "labels_df": labels_df
    }.items():

        if not isinstance(df, pd.DataFrame):
            raise TypeError(
                f"{name} must be a pandas DataFrame, "
                f"not {type(df).__name__}"
            )

        if df.empty:
            raise ValueError(f"{name} is empty")

        if "id" not in df.columns:
            raise KeyError(f"{name} does not contain an 'ID' column")

        if df["id"].isna().any():
            null_count = df["id"].isna().sum()
            raise ValueError(
                f"{name} contains {null_count} null ID value(s)"
            )

        if df["id"].duplicated().any():
            duplicate_ids = (
                df.loc[df["id"].duplicated(keep=False), "id"]
                .unique()
                .tolist()
            )

            raise ValueError(
                f"{name} contains duplicate IDs: {duplicate_ids}"
            )

    # Check the row counts.
    values_count = len(values_df)
    labels_count = len(labels_df)

    if values_count != labels_count:
        raise ValueError(
            f"Row count mismatch: values contains {values_count} rows, "
            f"but labels contains {labels_count} rows"
        )

    # Check that both frames contain exactly the same IDs.
    values_id_set = set(values_df["id"])
    labels_id_set = set(labels_df["id"])

    if values_id_set != labels_id_set:
        values_only = sorted(values_id_set - labels_id_set)
        labels_only = sorted(labels_id_set - values_id_set)

        raise ValueError(
            "ID sets do not match: "
            f"values-only IDs are {values_only}, "
            f"labels-only IDs are {labels_only}"
        )

    if allow_unordered:
        print("Values and labels data frames are compatible (ID order ignored)")
        return

    # Reset the indexes so that row positions can be compared reliably.
    values_ids = values_df["id"].reset_index(drop=True)
    labels_ids = labels_df["id"].reset_index(drop=True)

    # Check that the IDs match in the same order.
    mismatched_rows = values_ids.ne(labels_ids)

    if mismatched_rows.any():
        first_mismatch = mismatched_rows.idxmax()

        raise ValueError(
            f"ID mismatch at row {first_mismatch}: "
            f"values ID is {values_ids[first_mismatch]!r}, "
            f"but labels ID is {labels_ids[first_mismatch]!r}"
        )

    print("Values and labels data frames are compatible (IDs match in the same order)")

# Sanity check whether the input DF can be a valid 'labels' DF
def validate_labels_dataframe(labels_df):
    """Validate that labels_df has an ID column and at least one other (label) column."""

    if not isinstance(labels_df, pd.DataFrame):
        raise TypeError(
            f"labels_df must be a pandas DataFrame, "
            f"not {type(labels_df).__name__}"
        )

    if labels_df.empty:
        raise ValueError("labels_df must contain at least one row")

    if labels_df.columns.duplicated().any():
        duplicate_columns = (
            labels_df.columns[labels_df.columns.duplicated()]
            .unique()
            .tolist()
        )

        raise ValueError(
            f"labels_df contains duplicate columns: {duplicate_columns}"
        )

    if "id" not in labels_df.columns:
        raise KeyError("labels_df must contain an 'ID' column")

    if len(labels_df.columns) < 2:
        raise ValueError(
            "labels_df must contain at least one label column in addition to 'ID'"
        )

    if labels_df["id"].isna().any():
        null_count = labels_df["id"].isna().sum()

        raise ValueError(
            f"labels_df contains {null_count} null ID value(s)"
        )

    blank_ids = labels_df["id"].map(
        lambda value: isinstance(value, str) and not value.strip()
    )

    if blank_ids.any():
        raise ValueError(
            f"labels_df contains {blank_ids.sum()} blank ID value(s)"
        )

    if labels_df["id"].duplicated().any():
        duplicate_ids = (
            labels_df.loc[
                labels_df["id"].duplicated(keep=False),
                "id"
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            f"labels_df contains duplicate IDs: {duplicate_ids}"
        )

    print("Input was a valid Labels dataframe.")
    print(f"Size: {len(labels_df)} rows and {len(labels_df.columns) - 1} label column(s) (plus ID column)")
    
# Sanity check whether the input DF can be a valid 'values' DF
# Expect a list of expected column names as an input
def validate_values_dataframe(values_df, df_expected_columns):
    """Validate that values_df has an ID column and all columns in df_expected_columns"""

    
    if not isinstance(values_df, pd.DataFrame):
        raise TypeError(
            f"values_df must be a pandas DataFrame, "
            f"not {type(values_df).__name__}"
        )

    if values_df.empty:
        raise ValueError("values_df must contain at least one row")

    if values_df.columns.duplicated().any():
        duplicate_columns = (
            values_df.columns[values_df.columns.duplicated()]
            .unique()
            .tolist()
        )

        raise ValueError(
            f"values_df contains duplicate columns: {duplicate_columns}"
        )

    if "id" not in values_df.columns:
        raise KeyError("values_df must contain an 'id' column")

    missing_columns = []
    for col_name in df_expected_columns:
        if col_name not in values_df.columns:
            missing_columns.append(col_name)

    if len(missing_columns) > 0:
        raise ValueError(
            f"values_df is missing the following columns: {missing_columns}"
        )

    print("Input was a valid Values dataframe.")
    print(f"Size: {len(values_df)} rows and {len(values_df.columns) - 1} feature column(s) (plus ID column)")


In [58]:
# Do checks
column_names = [                   
                    'id',            'amount_tsh',         'date_recorded',
                'funder',            'gps_height',             'installer',
             'longitude',              'latitude',              'wpt_name',
           'num_private',                 'basin',            'subvillage',
                'region',           'region_code',         'district_code',
                   'lga',                  'ward',            'population',
        'public_meeting',           'recorded_by',     'scheme_management',
           'scheme_name',                'permit',     'construction_year',
       'extraction_type', 'extraction_type_group', 'extraction_type_class',
            'management',      'management_group',               'payment',
          'payment_type',         'water_quality',         'quality_group',
              'quantity',        'quantity_group',                'source',
           'source_type',          'source_class',       'waterpoint_type',
 'waterpoint_type_group']


print(">>> Validating training_set_values...")
validate_values_dataframe(training_set_values, column_names)
print()
print(">>> Validating test_set_values...")
validate_values_dataframe(test_set_values, column_names)
print()
print(">>> Validating training_set_labels...")
validate_labels_dataframe(training_set_labels)
print()
print(">>> Validating submission_format (should be labels-ilke)...")
validate_labels_dataframe(submission_format)
print()
print(">>> Confirming that training_set_values is compatible with training_set_labels")
compare_values_and_labels(training_set_values, training_set_labels)


print()
print(">>>Confirming that test_set_values is compatible with submission_format")
compare_values_and_labels(test_set_values, submission_format)


>>> Validating training_set_values...
Input was a valid Values dataframe.
Size: 59400 rows and 39 feature column(s) (plus ID column)

>>> Validating test_set_values...
Input was a valid Values dataframe.
Size: 14850 rows and 39 feature column(s) (plus ID column)

>>> Validating training_set_labels...
Input was a valid Labels dataframe.
Size: 59400 rows and 1 label column(s) (plus ID column)

>>> Validating submission_format (should be labels-ilke)...
Input was a valid Labels dataframe.
Size: 14850 rows and 1 label column(s) (plus ID column)

>>> Confirming that training_set_values is compatible with training_set_labels
Values and labels data frames are compatible (IDs match in the same order)

>>>Confirming that test_set_values is compatible with submission_format
Values and labels data frames are compatible (IDs match in the same order)


## 4. Take a first look

Start with a small sample, then produce a column-level summary containing:

- data type;
- non-null and null counts;
- percentage missing;
- number of distinct values;
- one or two example values.

A transposed sample may be less unpleasant than scrolling through forty columns.

In [59]:
# Display a small sample and create the column-level summary.


## 5. Audit the target

Count each `status_group` class, calculate its share of the training rows and add one readable chart.

Things to pin down:

- the majority-class accuracy a trivial classifier would achieve;
- whether accuracy alone would hide poor performance on a smaller class;
- which additional metric from the course would expose that weakness;
- whether later resampling must happen after the validation split.

In [60]:
# Calculate and plot the target distribution.
# Record the majority-class baseline and my metric choice.


## 6. Create a feature register

Work through the predictors in groups. I want one row per feature with these fields:

| Field | Purpose |
| --- | --- |
| Feature | Column name |
| Meaning | My plain-English interpretation |
| Review type | Numeric, categorical, binary, date, identifier/text or paired |
| Quality finding | Missingness, sentinels, range or cardinality issue |
| Relationship | Parent, child, duplicate concept or geographic pair |
| Target observation | Evidence of a relationship with `status_group` |
| Baseline decision to test | Keep, exclude, impute, group or derive |
| Confidence | High, medium or low, with a reason |

A DataFrame will help with sorting. A short Markdown table may survive better in the final write-up.

In [61]:
# Create the feature register in a form I will maintain.


### Review order

| Group | Features |
| --- | --- |
| Identifiers and text | `id`, `wpt_name`, `num_private` |
| Money and capacity | `amount_tsh`, `population` |
| Time | `date_recorded`, `construction_year` |
| Geography | `gps_height`, `longitude`, `latitude`, `basin`, `subvillage`, `region`, `region_code`, `district_code`, `lga`, `ward` |
| Organisations and permissions | `funder`, `installer`, `public_meeting`, `recorded_by`, `permit` |
| Scheme and management | `scheme_management`, `scheme_name`, `management`, `management_group` |
| Extraction | `extraction_type`, `extraction_type_group`, `extraction_type_class` |
| Payment | `payment`, `payment_type` |
| Water | `water_quality`, `quality_group`, `quantity`, `quantity_group` |
| Source | `source`, `source_type`, `source_class` |
| Waterpoint | `waterpoint_type`, `waterpoint_type_group` |

Start with identifiers and missing-value conventions. Leave the overlapping category thicket until the basic problems are visible.

## 7. Numeric feature template

Use the same routine for each numeric feature:

1. report count, missingness, distinct values, minimum, quartiles, maximum and mean;
2. count zeros and decide whether zero can represent a real measurement;
3. inspect a histogram and a box plot;
4. compare the distribution across target classes;
5. note skew, outliers and implausible ranges;
6. record a treatment to test. Do not change the column here.

Begin with `amount_tsh`, get the routine into decent shape, then reuse it. Zero is not missing merely because it is inconvenient.

In [62]:
# Write a reusable numeric-feature review.
# Apply it to one feature, interpret the output, then continue through the group.


## 8. Categorical feature template

Use the same routine for each categorical feature:

1. count distinct values, nulls and the most common levels;
2. calculate how much of the column the largest categories cover;
3. inspect rare levels and labels such as `none` or `unknown`;
4. compare target proportions within the most common levels;
5. check whether spelling, case or whitespace splits one category into several;
6. record whether to retain, group or exclude the feature in the baseline.

Limit plots to the most common levels. A forty-category legend is decorative fog.

In [63]:
# Write a reusable categorical-feature review.
# Choose the number of categories shown and explain that choice.


## 9. Binary feature template

Treat `public_meeting` and `permit` as binary fields with possible missing values.

Check value counts, nulls and class proportions. Missing and `False` mean different things, however convenient merging them might be.

In [64]:
# Review the binary features and record how I will represent missing values.


## 10. Date and construction-year template

Parse `date_recorded` and check the valid range, failed parses and number of records over time. Give `construction_year = 0` the suspicion it deserves.

A derived pump-age feature seems plausible. Check for negative ages and settle the unknown-year rule before using it.

In [65]:
# Parse and inspect the date fields.
# Explore pump age without adding it to the source DataFrame.


## 11. Identifier and free-text template

Measure uniqueness, repeated values and missingness for `id` and `wpt_name`. Decide whether each column describes the pump or identifies the row.

A row ID is not a useful feature merely because the model accepts numbers. Skip target charts with thousands of labels.

In [66]:
# Review identifier-like and free-text columns.
# Record the evidence for keeping or excluding each one from the baseline.


## 12. Geographic review

Longitude and latitude belong together. Check their ranges, count `(0, 0)` coordinates and plot the non-zero locations. Colouring by target may expose regional structure; it cannot explain it.

Compare the coordinates with `region`, `lga`, `ward` and the code fields. A random split may place neighbouring pumps on both sides, so note the validation risk.

In [67]:
# Audit the coordinate pair and compare it with the named geographic features.
# Record any validation concern raised by geographic clustering.


## 13. Related categorical hierarchies

Check whether each detailed value maps to one parent value in these families:

- `extraction_type` → `extraction_type_group` → `extraction_type_class`
- `management` → `management_group`
- `payment` → `payment_type`
- `water_quality` → `quality_group`
- `quantity` → `quantity_group`
- `source` → `source_type` → `source_class`
- `waterpoint_type` → `waterpoint_type_group`

Record inconsistencies and cardinality at each level. Start the baseline with one defensible level per concept. The entire hierarchy has to earn its place.

In [68]:
# Check the child-to-parent mappings and summarise each hierarchy.


## 14. Consolidate the missing-data audit

Bring the feature reviews together and separate:

- explicit nulls recognised by the DataFrame;
- numeric sentinels such as zero where zero is implausible;
- categorical sentinels such as `none` or `unknown`;
- genuine zero and `False` values.

Use one proposed rule per affected column. No grand replace-all-zeros manoeuvre.

In [69]:
# Summarise explicit and suspected missing values by column.
# Add my proposed treatment and the evidence supporting it.


## 15. Compare training and test predictors

Compare shape, missingness, numeric ranges and category levels. Count categories that appear in the test set but not the training set.

Use the result to design the encoder and pipeline. The test file has no target and gets no vote on model performance.

In [70]:
# Compare training and test predictor distributions and category coverage.


## 16. Check duplicates and consistency

Look for duplicate rows with and without `id`, impossible values, conflicting category mappings and columns with one value.

Classify each result as a data error, a legitimate repeated observation or a feature with no predictive information.

In [71]:
# Run duplicate, constant-column and consistency checks.


## 17. Finish the decision log

Summarise the choices that notebook 02 must implement and test. If a decision has no evidence here, it is not ready:

| Issue | Evidence | Proposed treatment | Validation check | Confidence |
| --- | --- | --- | --- | --- |
| Target imbalance |  |  |  |  |
| Explicit missing values |  |  |  |  |
| Hidden missing-value markers |  |  |  |  |
| High-cardinality features |  |  |  |  |
| Related category hierarchies |  |  |  |  |
| Date-derived features |  |  |  |  |
| Geographic features |  |  |  |  |
| Train/test category differences |  |  |  |  |

Add rows for findings that do not fit these headings.

In [72]:
# Create a programmatic summary if it helps carry decisions into notebook 02.


## 18. Bridge to the taught modelling workflow

Keep the course sequence visible in notebook 02:

1. choose the validation measure and create a reproducible stratified split;
2. establish a majority-class reference and an interpretable decision-tree baseline;
3. place imputation and encoding inside the training pipeline;
4. test oversampling on training data only;
5. use k-fold cross-validation for model comparison;
6. tune the strongest taught model after the baseline works.

Methods beyond the course can wait until this sequence works. Each addition needs a problem from the audit and a repeatable validation improvement; novelty alone is not a result.

## Completion checklist

- [ ] All four files pass the structure and identifier checks.
- [ ] The target balance and majority baseline are recorded.
- [ ] Each predictor has an entry in the feature register.
- [ ] Each suspected missing-value marker has evidence and a proposed treatment.
- [ ] Related categorical and geographic features have paired checks.
- [ ] Training/test differences inform the preprocessing plan.
- [ ] Every baseline decision points back to an audit finding.
- [ ] The source DataFrames remain unchanged.